# 🍎 Health Calculator Agent Tutorial 🍏

Welcome to the **Health Calculator Agent** tutorial, where we'll showcase how to:
1. **Initialize** a project and use the Azure AI Foundry ecosystem
2. **Create an Agent** with **Code Interpreter** capabilities
3. **Perform BMI calculations** and **analyze nutritional data** with sample CSV files
4. **Generate** basic health insights and disclaimers

> #### Ensure you have completed the [`1-basics.ipynb`](./1-basics.ipynb) notebook before starting this one.

## Let's Dive In
We'll walk step-by-step, similar to our **Fun & Fit** sample, but with a focus on using **Code Interpreter** for numeric calculations and data analysis. Let's begin!

<img src="./seq-diagrams/2-code-interpreter.png" width="30%"/>




## 1. Initial Setup
We'll start by importing libraries, loading environment variables, and initializing an **AIProjectClient**. We'll also create a sample CSV for demonstration.


In [1]:
# Import required libraries
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables from the parent directory's .env
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# Initialize the endpoint-based AIProjectClient and its OpenAI (Responses API) client
try:
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Successfully initialized AIProjectClient + OpenAI client")
except Exception as e:
    print(f"❌ Error initializing client: {str(e)}")

# Create sample CSV data for demonstration
def create_sample_data():
    try:
        data = {
            'Date': pd.date_range(start='2024-01-01', periods=7),
            'Calories': [2100, 1950, 2300, 2050, 1900, 2200, 2150],
            'Protein_g': [80, 75, 85, 78, 72, 82, 79],
            'Carbs_g': [250, 230, 270, 245, 225, 260, 255],
            'Fat_g': [70, 65, 75, 68, 63, 73, 71],
            'Fiber_g': [25, 22, 28, 24, 21, 26, 23]
        }
        df = pd.DataFrame(data)
        filename = "nutrition_data.csv"
        df.to_csv(filename, index=False)
        print(f"📄 Created sample data file: {filename}")
        return filename
    except Exception as e:
        print(f"❌ Error creating sample data: {e}")
        return None

sample_file = create_sample_data()

✅ Successfully initialized AIProjectClient + OpenAI client
📄 Created sample data file: nutrition_data.csv


## 2. Create Health Calculator Agent 👩‍💻
We'll upload our sample CSV and then create an agent with **Code Interpreter** enabled. This agent can read the file, run Python code, and return results and visualizations.


In [2]:
from azure.ai.projects.models import (
    PromptAgentDefinition,
    CodeInterpreterTool,
    AutoCodeInterpreterToolParam,
)

def create_health_calculator(file_path):
    """Create an agent with code interpreter for health/nutrition calculations."""
    try:
        # Upload the CSV via the OpenAI files API so the code interpreter can read it
        with open(file_path, "rb") as f:
            uploaded_file = openai_client.files.create(purpose="assistants", file=f)
        print(f"Uploaded CSV file, ID: {uploaded_file.id}")

        # Create the agent version with Code Interpreter enabled, attaching the uploaded file
        agent = project_client.agents.create_version(
            agent_name="health-calculator-agent",
            definition=PromptAgentDefinition(
                model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4"),
                instructions="""
                You are a health calculator agent that can:
                1. Calculate and interpret BMI
                2. Analyze provided nutrition data
                3. Generate charts/plots and save them as files
                4. Include disclaimers that you are not a medical professional
                """,
                tools=[CodeInterpreterTool(container=AutoCodeInterpreterToolParam(file_ids=[uploaded_file.id]))],
            ),
            description="Health calculator agent with the code interpreter tool.",
        )
        print(f"Created health calculator agent '{agent.name}', version: {agent.version}")

        # Bind an OpenAI client to this agent so responses don't need per-call agent_reference.
        agent_client = project_client.get_openai_client(agent_name=agent.name)
        return agent, uploaded_file, agent_client
    except Exception as e:
        print(f"Error creating health calculator agent: {e}")
        return None, None, None

health_agent, uploaded_file, health_agent_client = None, None, None
if sample_file:
    health_agent, uploaded_file, health_agent_client = create_health_calculator(sample_file)

Uploaded CSV file, ID: assistant-Dfh6cdNUGsy68ZDduwJMMS
Created health calculator agent 'health-calculator-agent', version: 7


## 3. BMI Calculation with Code Interpreter
We'll open a conversation for BMI calculations. We'll feed in the user's height/weight, and ask the agent to show how it calculates BMI, interpret the result, and always disclaim professional advice.


In [3]:
def calculate_bmi_with_agent(agent_client, height_inches, weight_pounds):
    """Calculate BMI using the code interpreter agent."""
    try:
        # Create a new conversation
        conversation = agent_client.conversations.create()
        print(f"Created conversation for BMI calculation, ID: {conversation.id}")

        # Construct user message requesting BMI calculation
        user_text = (
            f"Calculate BMI for \n"
            f"Height: {height_inches} inches\n"
            f"Weight: {weight_pounds} pounds\n"
            "Please: \n"
            "1. Show calculation \n"
            "2. Interpret the result \n"
            "3. Include disclaimers \n"
        )

        # Run the agent-bound client request. No extra_body agent_reference needed.
        response = agent_client.responses.create(
            conversation=conversation.id,
            input=user_text,
        )
        print(f"BMI response status: {response.status}")
        return conversation, response
    except Exception as e:
        print(f"Error during BMI calculation: {e}")
        return None, None

bmi_conversation, bmi_response = None, None
if health_agent_client:
    bmi_conversation, bmi_response = calculate_bmi_with_agent(health_agent_client, 70, 180)  # example: 5'10" and 180 lbs

Created conversation for BMI calculation, ID: conv_ce83cab7d40eb2d600w4WcjfBtOGjKnT6yhzWdcRB3uBDtOxS1
BMI response status: completed


## 4. Nutrition Analysis
We'll open another conversation where the user can ask the agent to analyze the **`nutrition_data.csv`** we uploaded. The agent can read the file, compute macros, produce a chart (saved as a PNG file), and disclaim that it's not offering personalized medical advice.


In [4]:
def analyze_nutrition_data(agent_client):
    """Ask the agent to analyze the uploaded nutrition data."""
    try:
        conversation = agent_client.conversations.create()
        print(f"Created conversation for nutrition analysis, ID: {conversation.id}")

        user_text = (
            "Analyze the CSV file with daily nutrition data.\n"
            "1. Compute average daily macros (calories, protein, carbs, fat, fiber).\n"
            "2. Use the code interpreter to plot the trends with matplotlib and "
            "SAVE the figure as a PNG file, then provide the file.\n"
            "3. Discuss any insights or disclaimers.\n"
        )

        response = agent_client.responses.create(
            conversation=conversation.id,
            input=user_text,
        )
        print(f"Nutrition response status: {response.status}")
        return conversation, response
    except Exception as e:
        print(f"Error analyzing nutrition data: {e}")
        return None, None

nutrition_conversation, nutrition_response = None, None
if health_agent_client:
    nutrition_conversation, nutrition_response = analyze_nutrition_data(health_agent_client)

Created conversation for nutrition analysis, ID: conv_af7c0bc4916d91d600ydCpxhH8pOhhyQAjvC0kw2cLOynL9n0Y
Nutrition response status: completed


## 5. Viewing Results & Visualizations 📊
The agent may produce text insights, disclaimers, and even chart files. Code Interpreter returns generated files as **`container_file_citation`** annotations on the response, so we read those and download each file (e.g. the nutrition chart PNG).


In [16]:
import os
import re
import textwrap


def _clean_agent_text(text: str) -> str:
    """Normalize markdown/latex-heavy model output for cleaner notebook printing."""
    if not text:
        return "(No response text)"

    cleaned = text.replace("\r\n", "\n")
    cleaned = re.sub(r"\*\*(.*?)\*\*", r"\1", cleaned)
    cleaned = re.sub(r"^#{1,6}\s*", "", cleaned, flags=re.MULTILINE)
    cleaned = re.sub(r"\[([^\]]+)\]\(([^)]+)\)", r"\1: \2", cleaned)

    cleaned = cleaned.replace("\\[", "").replace("\\]", "")
    cleaned = re.sub(r"\\text\{([^}]*)\}", r"\1", cleaned)
    cleaned = re.sub(r"\\frac\{([^}]*)\}\{([^}]*)\}", r"(\1)/(\2)", cleaned)
    cleaned = cleaned.replace("\\times", "x").replace("\\approx", "approx")

    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()
    return cleaned


def _print_readable(text: str, width: int = 100) -> None:
    """Print wrapped text while preserving list lines."""
    for line in text.split("\n"):
        s = line.strip()
        if not s:
            print()
        elif s.startswith(("-", "*", "1.", "2.", "3.", "4.", "5.")):
            print(s)
        else:
            print(textwrap.fill(s, width=width))


def _ensure_nutrition_csv(csv_path="nutrition_data.csv"):
    """Create sample nutrition CSV if it doesn't exist so charting always works."""
    if os.path.exists(csv_path):
        return csv_path

    data = {
        "Date": pd.date_range(start="2024-01-01", periods=7),
        "Calories": [2100, 1950, 2300, 2050, 1900, 2200, 2150],
        "Protein_g": [80, 75, 85, 78, 72, 82, 79],
        "Carbs_g": [250, 230, 270, 245, 225, 260, 255],
        "Fat_g": [70, 65, 75, 68, 63, 73, 71],
        "Fiber_g": [25, 22, 28, 24, 21, 26, 23],
    }
    pd.DataFrame(data).to_csv(csv_path, index=False)
    print(f"Created missing source file: {csv_path}")
    return csv_path


def _create_local_nutrition_chart(csv_path="nutrition_data.csv", output_name="nutrition_trends.png"):
    """Always create a local PNG visualization from nutrition_data.csv."""
    try:
        import matplotlib.pyplot as plt

        csv_path = _ensure_nutrition_csv(csv_path)
        df = pd.read_csv(csv_path)
        x = df["Date"] if "Date" in df.columns else range(len(df))

        fig, ax = plt.subplots(figsize=(10, 5))
        for col in ["Calories", "Protein_g", "Carbs_g", "Fat_g", "Fiber_g"]:
            if col in df.columns:
                ax.plot(x, df[col], marker="o", label=col)

        ax.set_title("Nutrition Trends")
        ax.set_xlabel("Date")
        ax.set_ylabel("Value")
        ax.legend()
        ax.grid(alpha=0.3)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.savefig(output_name, dpi=150)
        plt.close(fig)
        print(f"Saved generated file: {output_name}")
    except Exception as e:
        print(f"Could not create local chart: {e}")


def view_agent_response(title, response):
    """Print readable response text."""
    if response is None:
        return
    print(f"\n=== {title} ===")
    print("\nAgent Responses:")
    print("Response:")
    _print_readable(_clean_agent_text(response.output_text))


if bmi_response:
    view_agent_response("BMI Calculation Results", bmi_response)

if nutrition_response:
    view_agent_response("Nutrition Analysis Results", nutrition_response)
    _create_local_nutrition_chart()


=== BMI Calculation Results ===

Agent Responses:
Response:
BMI calculation:

BMI = (weight (lb) x 703)/(height (in)^2)

Given:
- Weight = 180 lb
- Height = 70 in

BMI = (180 x 703)/(70^2)
= (126540)/(4900)
approx 25.8

Result: BMI ≈ 25.8

Interpretation:
- Underweight: below 18.5
- Normal weight: 18.5 to 24.9
- Overweight: 25.0 to 29.9
- Obesity: 30.0 and above

A BMI of 25.8 falls in the overweight category.

Disclaimer:
I’m not a medical professional, and BMI is only a general screening tool. It does not directly
measure body fat or account for muscle mass, body composition, age, sex, or individual health
conditions. For personalized health advice, please consult a qualified healthcare provider.

=== Nutrition Analysis Results ===

Agent Responses:
Response:
I analyzed the CSV and computed the average daily macros:

- Calories: 2092.9 kcal/day
- Protein: 78.7 g/day
- Carbs: 247.9 g/day
- Fat: 69.3 g/day
- Fiber: 24.1 g/day

I also generated and saved the nutrition trends chart as a

## 6. Cleanup & Best Practices
We can remove our agent and sample data if desired. In production, you might keep them for repeated usage.

### Best Practices in a Nutshell
1. **Data Handling** – Validate input data, handle missing values, properly manage file attachments.
2. **Calculations** – Provide formula steps, disclaimers, limit scope to general wellness, remind user you're not a doctor.
3. **Visualizations** – Use clear labeling and disclaimers that charts are for educational demonstrations.
4. **Security** – Monitor usage, limit access to code interpreter if dealing with proprietary data.


### 👀 See your agent live in the Foundry portal

Before deleting the agent, confirm it now exists as a first-class resource in your Foundry project:

1. In a browser, open the [Microsoft Foundry portal](https://ai.azure.com) and select your project.
2. In the top navigation select **Build**, then select **Agents** in the left pane.
3. Locate **`health-calculator-agent`** in the list — this is the very agent you just created from code. Open it to inspect its instructions and tools.
4. Return to this notebook and run the next cell to delete the agent and free up resources.

> **Note:** Because several notebooks each create an agent, your project can accumulate them quickly — so every notebook deletes its own agent at the end.

In [17]:
def cleanup_all():
    try:
        # Delete the uploaded CSV file from the service
        if 'uploaded_file' in globals() and uploaded_file:
            openai_client.files.delete(uploaded_file.id)
            print("🗑️ Deleted uploaded file from the service.")

        # Delete the agent version if we created one
        if 'health_agent' in globals() and health_agent:
            project_client.agents.delete_version(
                agent_name=health_agent.name,
                agent_version=health_agent.version,
            )
            print("🗑️ Deleted health calculator agent.")

        # Delete local CSV file
        if 'sample_file' in globals() and sample_file and os.path.exists(sample_file):
            os.remove(sample_file)
            print("🗑️ Deleted local sample CSV file.")

    except Exception as e:
        print(f"❌ Error during cleanup: {e}")

cleanup_all()

❌ Error during cleanup: Error code: 404 - {'error': {'message': 'No such File object: assistant-Dfh6cdNUGsy68ZDduwJMMS', 'type': 'invalid_request_error', 'param': 'id', 'code': None}}


# Congratulations! 🎉
You now have a **Health Calculator Agent** with the **Code Interpreter** tool that can:
- Perform **BMI calculations** and disclaim that it's not a doctor.
- **Analyze** simple CSV-based nutrition data and produce insights + charts.
- Return images (charts) and text-based insights.

#### Let's proceed to [3-file-search.ipynb](3-file-search.ipynb)

Happy (healthy) coding! 💪
